In [1]:
from jetbot import Camera, bgr8_to_jpeg, Robot
import jetson_inference
import jetson_utils
import ipywidgets.widgets as widgets
from IPython.display import display
import numpy as np
import cv2

# Load model + robot
net = jetson_inference.detectNet("ssd-mobilenet-v2", threshold=0.3)

# Camera + widgets
FRAME_W, FRAME_H = 300, 300

camera = Camera.instance(width=FRAME_W, height=FRAME_H)
image_widget = widgets.Image(format='jpeg', width=FRAME_W, height=FRAME_H)

status_label       = widgets.Label(value="Idle")
speed_slider       = widgets.FloatSlider(description='speed',       min=0.0, max=0.4, value=0.15, step=0.01)
turn_gain_slider   = widgets.FloatSlider(description='turn gain',   min=0.0, max=2.0, value=1.0,  step=0.1)
target_size_slider = widgets.FloatSlider(description='target size', min=0.3, max=1.0, value=0.8,  step=0.05)
enable_button      = widgets.ToggleButton(value=False, description='🚗 Enable Robot',
                                          button_style='danger')
robot = Robot()

display(widgets.VBox([
    image_widget, status_label,
    speed_slider, turn_gain_slider, target_size_slider,
    enable_button,
]))

# COCO classes we treat as "cans/bottles"
TARGET_CLASSES = {44: "bottle", 47: "cup"}

def process_frame(change):
    frame = change['new']  # BGR numpy array from jetbot camera
    h, w = frame.shape[:2]
    
    # BGR → RGBA for jetson-inference
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[:, :, 0] = frame[:, :, 2]
    rgba[:, :, 1] = frame[:, :, 1]
    rgba[:, :, 2] = frame[:, :, 0]
    rgba[:, :, 3] = 255
    
    cuda_img = jetson_utils.cudaFromNumpy(rgba)
    detections = net.Detect(cuda_img, overlay="none")
    targets = [d for d in detections if d.ClassID in TARGET_CLASSES]
    
    output = frame.copy()
    
    # Pick the BIGGEST target (= closest)
    target = max(targets, key=lambda d: d.Width * d.Height) if targets else None
    
    if target is not None:
        x1, y1, x2, y2 = int(target.Left), int(target.Top), int(target.Right), int(target.Bottom)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
        box_w, box_h = x2 - x1, y2 - y1
        size_ratio = max(box_w / w, box_h / h)  # whichever dimension is bigger
        
        # Draw target overlay
        label = f"{TARGET_CLASSES[target.ClassID]} {target.Confidence:.2f} size={size_ratio:.0%}"
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output, label, (x1, max(y1 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        cv2.line(output, (w//2, 0), (w//2, h), (255, 0, 0), 1)        # frame center
        cv2.circle(output, (cx, cy), 5, (0, 0, 255), -1)              # box center
        
        # ===== CONTROL LOGIC =====
        target_size = target_size_slider.value
        base_speed  = speed_slider.value
        turn_gain   = turn_gain_slider.value
        
        if size_ratio >= target_size:
            # Arrived → stop
            if enable_button.value:
                robot.stop()
            status_label.value = f"✅ ARRIVED! Box fills {size_ratio:.0%}"
        else:
            # Steering: how far off-center is the box? (-1 = far left, +1 = far right)
            error = (cx - w/2) / (w/2)
            
            # Small deadzone — ignore tiny errors to prevent wobble
            if abs(error) < 0.08:
                error = 0
            
            steering = error * turn_gain * base_speed
            left_motor  = max(min(base_speed + steering,  1.0), -1.0)
            right_motor = max(min(base_speed - steering,  1.0), -1.0)
            
            if enable_button.value:
                robot.set_motors(left_motor, right_motor)
            
            status_label.value = (f"🎯 Tracking | size={size_ratio:.0%} "
                                  f"| error={error:+.2f} | L={left_motor:.2f} R={right_motor:.2f}")
    else:
        # No target → stop
        if enable_button.value:
            robot.stop()
        status_label.value = "🔍 No target in view"
    
    image_widget.value = bgr8_to_jpeg(output)

camera.observe(process_frame, names='value')
print("✅ Running! Toggle '🚗 Enable Robot' when ready.")

RuntimeError: Could not initialize camera.  Please see error trace.

In [ ]:
from jetbot import Camera, bgr8_to_jpeg

# Camera + display widgets (RGB feed + depth map side by side)
camera = Camera.instance(width=300, height=300)
rgb_widget   = widgets.Image(format='jpeg', width=300, height=300)
depth_widget = widgets.Image(format='jpeg', width=300, height=300)
display(widgets.HBox([rgb_widget, depth_widget]))


In [ ]:
camera.unobserve(process_frame, names='value')
import time; time.sleep(0.1)
camera.stop()